## Initialization

In [ ]:
from torch.func import vjp
import torch
from torch.func import jacrev, functional_call
import torch.nn as nn
from torch import Tensor

import torch.nn.functional as F

import sys
import os

current_notebook_dir = os.path.dirname(os.path.abspath('__file__'))
project_root_dir = os.path.abspath(os.path.join(current_notebook_dir, '../../'))

# 将这个父目录添加到sys.path的最前面
if project_root_dir not in sys.path:
    sys.path.insert(0, project_root_dir)

print(sys.path)

In [ ]:
from loss_distribution.pytorch_script.visual_utils \
	import load_cifar10_data, load_model_state_dict

from ntk_result.trials.utils import *

import torchvision
import torchvision.transforms as transforms

data_pth = '/home/hqdeng7/lijuyang/generalization/loss_distribution/pytorch_script/data/cifar10'
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])
train_ds = torchvision.datasets.CIFAR10(root=data_pth, train=True, download=True, transform=transform)
test_ds = torchvision.datasets.CIFAR10(root=data_pth, train=False, download=True, transform=transform)

In [ ]:
model200_path = '/home/hqdeng7/lijuyang/generalization/loss_distribution/model_training_results/cifar10_resnet20/model_200.pth'
model200 = load_model_state_dict('cifar10', 'resnet20', 10, model200_path, 'cuda')
model100_path = '/home/hqdeng7/lijuyang/generalization/loss_distribution/model_training_results/cifar10_resnet20/model_100.pth'
model100 = load_model_state_dict('cifar10', 'resnet20', 10, model100_path, 'cuda')

## Compute all grads

In [ ]:
from tqdm import tqdm
from torch.utils.data import DataLoader, Dataset
from torch.func import functional_call, jacrev

# 假设您已经定义了 compute_param_gradients 和 flatten_grads_dict
# ...

# def get_all_gradients(model: nn.Module, ds: Dataset, batch_size: int = 32, device='cuda'):
#     model.to(device)
#     model.eval()

#     dl = DataLoader(ds, batch_size=batch_size, shuffle=False)
#     all_grads = []

#     for inputs, labels in tqdm(dl):
#         grads_per_sample_dict = compute_param_gradients(model, inputs, labels, device=device)
        
#         flattened_grad = flatten_grads_dict(grads_per_sample_dict)
#         all_grads.append(flattened_grad)
#         torch.cuda.empty_cache()

#     all_grads_tensor = torch.cat(all_grads, dim=0)

#     return all_grads_tensor

In [ ]:
from tqdm import tqdm
from torch.utils.data import DataLoader, Dataset
from torch.func import functional_call, jacrev

# def compute_all_grads(model: nn.Module, ds: Dataset, device='cuda', batch_size: int = 64):
#     """
#     使用 torch.func.vmap 向量化地获取所有样本的梯度。
#     """
#     model.to(device)
#     model.eval()

#     params = dict(model.named_parameters())
#     buffers = dict(model.named_buffers())

#     dl = DataLoader(ds, batch_size=batch_size, shuffle=False)

#     # 将 compute_single_gradient 定义为内部函数，以便访问 model
#     def compute_single_gradient(p, b, i, l):
#         outputs = functional_call(model, (p, b), i.unsqueeze(0))
#         losses = nn.CrossEntropyLoss(reduction='none')(outputs, l.unsqueeze(0))
#         return losses

#     # vmap 向量化 jacrev，用于批量计算
#     grads_fn = jacrev(compute_single_gradient, argnums=0)
#     vmap_grads_fn = torch.func.vmap(grads_fn, in_dims=(None, None, 0, 0))

#     all_grads = []

#     for inputs, labels in tqdm(dl):
#         inputs, labels = inputs.to(device), labels.to(device)
        
#         # 使用 vmap_grads_fn 进行高效的批量梯度计算
#         grads_per_sample_dict = vmap_grads_fn(params, buffers, inputs, labels)
        
#         flattened_grad = flatten_grads_dict(grads_per_sample_dict)
#         all_grads.append(flattened_grad)
        
#         # 释放缓存以防万一
#         torch.cuda.empty_cache()

#     all_grads_tensor = torch.cat(all_grads, dim=0)

#     return all_grads_tensor

def compute_all_grads(model: nn.Module, ds: Dataset, device='cuda', batch_size: int = 64):
    """
    使用 torch.func.vmap 向量化地获取所有样本的梯度。
    """

    dl = DataLoader(ds, batch_size=batch_size, shuffle=False)

    all_grads = []

    for inputs, labels in tqdm(dl):
        all_grads.append(compute_param_grads(model100, inputs, labels))
    all_grads_tensor = torch.cat(all_grads, dim=0)

    return all_grads_tensor

In [ ]:
all_grads = compute_all_grads(model100, test_ds)

In [ ]:
all_grads.requires_grad

In [ ]:
import numpy as np
from sklearn.decomposition import PCA
from sklearn.cluster import MiniBatchKMeans
import matplotlib.pyplot as plt
from sklearn.cluster import Birch
from sklearn.preprocessing import StandardScaler

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(all_grads.tolist())

# 3. 使用PCA进行降维
# 选择保留95%的方差，或者指定降维后的维度数量
# 这里我们选择保留95%的方差
print("\n3. 使用PCA进行降维...")
pca = PCA(n_components=0.95, random_state=42)
X_pca = pca.fit_transform(X_scaled)
print(f"降维后的数据形状: {X_pca.shape}")

# 4. 使用MiniBatchKMeans进行聚类
# MiniBatchKMeans比传统的KMeans更适合处理大规模数据，因为它每次只使用一小部分数据进行更新
print("\n4. 使用MiniBatchKMeans进行聚类...")
n_clusters = 50  # 假设我们想将数据聚成5类
kmeans = MiniBatchKMeans(n_clusters=n_clusters, random_state=42, n_init=10)
kmeans.fit(X_pca)

# 5. 获取聚类结果
labels = kmeans.labels_
print("\n5. 聚类完成！")
print(f"聚类结果（前20个样本的标签）: {labels[:20]}")
print(f"每个聚类的样本数: {[np.sum(labels == i) for i in range(n_clusters)]}")

In [ ]:
import cupy as cp
from cuml.decomposition import PCA as GPU_PCA
from cuml.cluster import KMeans as GPU_KMeans


# 步骤 1：PCA 降维（建议降到 100~512 维）
print("Running GPU PCA...")
pca = GPU_PCA(n_components=100)
X_reduced = pca.fit_transform(all_grads)
print("PCA done. Reduced shape:", X_reduced.shape)

# 步骤 2：GPU KMeans 聚类
print("Running GPU KMeans...")
n_clusters = 10
kmeans = GPU_KMeans(n_clusters=n_clusters, init="k-means||", max_iter=300)
kmeans.fit(X_reduced)

# 获取聚类标签
labels = kmeans.labels_

print("Clustering done. Labels shape:", labels.shape)

## high test loss cluster

In [ ]:
model100_losses_fn = get_batch_loss_fn(model100)
hloss_samples = {}
hloss_samples[('test', 'epoch100', 'k500')] =find_topk_samples(test_ds, fn=model100_losses_fn, k=500)

In [ ]:
hloss_samples_losses = {key: np.array(list(zip(*value))[0]) for key, value in hloss_samples.items()}
hloss_samples_indices = {key: np.array(list(zip(*value))[1]) for key, value in hloss_samples.items()}	

In [ ]:
from torch.utils.data import Subset
hloss_sample_grads = {}
hloss_sample_grads[('test', 'epoch100', 'k500')] = \
	compute_all_grads(model100, Subset(test_ds, hloss_samples_indices[('test', 'epoch100', 'k500')]))

In [ ]:
import numpy as np
from sklearn.decomposition import PCA
from sklearn.cluster import MiniBatchKMeans
import matplotlib.pyplot as plt
from sklearn.cluster import Birch
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(hloss_sample_grads[('test', 'epoch100', 'k500')].tolist())



In [ ]:
def dim_reduc_cluster(n_components, n_clusters):
	print("\n使用PCA进行降维...")
	pca = PCA(n_components=0.5, random_state=42)
	X_pca = pca.fit_transform(X_scaled)
	print(f"降维后的数据形状: {X_pca.shape}")

	# MiniBatchKMeans比传统的KMeans更适合处理大规模数据，因为它每次只使用一小部分数据进行更新
	print("\n使用MiniBatchKMeans进行聚类...")
	n_clusters = n_clusters
	kmeans = MiniBatchKMeans(n_clusters=n_clusters, random_state=42, n_init=10)
	kmeans.fit(X_pca)

	# 获取聚类结果
	labels = kmeans.labels_
	print("\n聚类完成！")
	print(f"每个聚类的样本数: {[np.sum(labels == i) for i in range(n_clusters)]}")

	return labels

In [ ]:
cls15_labels = dim_reduc_cluster(500, 15)

In [ ]:
def plot_cluster(labels, cluster_idx, len=4):
	fig, axes = plt.subplots(len, len, figsize=(16, 16))
	axes = axes.flatten()
	cluster_sample_indices = hloss_samples_indices[('test', 'epoch100', 'k500')][np.array(labels) == cluster_idx]
	for i in range(len*len):
		img = inverse_trans_cifar10(test_ds[cluster_sample_indices[i]][0])
		img_show(img, ax=axes[i])		


In [ ]:
plot_cluster(cls15_labels, 0, len=5)

In [ ]:
plot_cluster(cls15_labels, 1, len=5)

In [ ]:
plot_cluster(cls15_labels, 2, len=4)

In [ ]:
plot_cluster(cls15_labels, 3, len=4)

In [ ]:
plot_cluster(cls15_labels, 14, len=7)

In [ ]:
plot_cluster(cls15_labels, 13, len=5)

In [ ]:
cls30_labels = dim_reduc_cluster(500, 30)

In [ ]:
plot_cluster(cls30_labels, 0, len=7)

In [ ]:
plot_cluster(cls30_labels, 15)

In [ ]:
plot_cluster(cls30_labels, 13)

In [ ]:
plot_cluster(cls30_labels, 17, len=3)

In [ ]:
plot_cluster(cls30_labels, 27, len=4)

In [ ]:
plot_cluster(cls30_labels, 28, len=4)

### peek the grad dict

In [ ]:
def compute_param_grads_dict(model: nn.Module, 
						inputs: torch.Tensor,
						labels: torch.Tensor,
						loss_fn = nn.CrossEntropyLoss(reduction='mean'),
						device='cuda'):
	"""
	使用 torch.func.vmap 向量化地获取所有样本的梯度。
	"""
	loss_fn = loss_fn if loss_fn.reduction == 'mean' else type(loss_fn)(reduction='mean')

	model.to(device)
	model.eval()

	params = dict(model.named_parameters())
	buffers = dict(model.named_buffers())

	# 将 compute_single_gradient 定义为内部函数，以便访问 model
	def compute_single_gradient(params, buffers, input, label):
		output = functional_call(model, (params, buffers), input.unsqueeze(0))
		loss = loss_fn(output, label.unsqueeze(0))
		return loss

	# vmap 向量化 jacrev，用于批量计算
	grads_fn = torch.func.grad(compute_single_gradient, argnums=0)
	vmap_grads_fn = torch.func.vmap(grads_fn, in_dims=(None, None, 0, 0))

	inputs, labels = inputs.to(device), labels.to(device)
	
	# 使用 vmap_grads_fn 进行高效的批量梯度计算
	grads_per_sample_dict = vmap_grads_fn(params, buffers, inputs, labels)
	
	# 释放缓存以防万一
	torch.cuda.empty_cache()

	return grads_per_sample_dict

In [ ]:
input, label = test_ds[hloss_samples_indices[('test', 'epoch100', 'k500')][2]]
input_, label_ = input.unsqueeze(0), torch.tensor(label).unsqueeze(0)
compute_param_grads_dict(model100, input_, label_)

In [ ]:
hloss_samples_indices[('test', 'epoch100', 'k500')]

## training clusters from test

In [120]:
cls_indices_arrs = [hloss_samples_indices[('test', 'epoch100', 'k500')][cls30_labels == i] for i in range(30)]
cls_indices_arrs = [arr for arr in cls_indices_arrs if len(arr) >= 8]

In [122]:
tar_indices = [arr[0] for arr in cls_indices_arrs]